In [0]:
# Create our project database
spark.sql("CREATE DATABASE IF NOT EXISTS ecommerce.youth_employment")
print("Database created ✓")

Database created ✓


In [0]:
# Load directly from existing Delta tables
from pyspark.sql import functions as F

unemp_raw = spark.table("ecommerce.default.unemployment_rate_upto_11_2020")
jobs_raw  = spark.table("ecommerce.default.jobs_in_data")

print(f"Unemployment rows : {unemp_raw.count():,}")
print(f"Jobs rows         : {jobs_raw.count():,}")
print("\nUnemployment columns:", unemp_raw.columns)
print("\nJobs columns:", jobs_raw.columns)

Unemployment rows : 267
Jobs rows         : 9,355

Unemployment columns: ['Region', ' Date', ' Frequency', ' Estimated Unemployment Rate (%)', ' Estimated Employed', ' Estimated Labour Participation Rate (%)', 'longitude', 'latitude']

Jobs columns: ['work_year', 'job_title', 'job_category', 'salary_currency', 'salary', 'salary_in_usd', 'employee_residence', 'experience_level', 'employment_type', 'work_setting', 'company_location', 'company_size']


In [0]:
# Clean unemployment table
unemp_clean = unemp_raw.toDF(
    "region", "date", "frequency",
    "unemployment_rate", "estimated_employed",
    "labour_participation_rate",
    "longitude", "latitude"
).withColumn("region", F.trim(F.col("region"))) \
 .withColumn("unemployment_rate", F.col("unemployment_rate").cast("double")) \
 .withColumn("labour_participation_rate", F.col("labour_participation_rate").cast("double")) \
 .withColumn("estimated_employed", F.col("estimated_employed").cast("long"))

print("Unemployment cleaned ✓")
unemp_clean.show(5, truncate=False)

Unemployment cleaned ✓
+--------------+-----------+---------+-----------------+------------------+-------------------------+---------+--------+
|region        |date       |frequency|unemployment_rate|estimated_employed|labour_participation_rate|longitude|latitude|
+--------------+-----------+---------+-----------------+------------------+-------------------------+---------+--------+
|Andhra Pradesh| 31-01-2020| M       |5.48             |16635535          |41.02                    |15.9129  |79.74   |
|Andhra Pradesh| 29-02-2020| M       |5.83             |16545652          |40.9                     |15.9129  |79.74   |
|Andhra Pradesh| 31-03-2020| M       |5.79             |15881197          |39.18                    |15.9129  |79.74   |
|Andhra Pradesh| 30-04-2020| M       |20.51            |11336911          |33.1                     |15.9129  |79.74   |
|Andhra Pradesh| 31-05-2020| M       |17.43            |12988845          |36.46                    |15.9129  |79.74   |
+--------

In [0]:
# Clean jobs table
jobs_clean = jobs_raw \
    .withColumn("salary_in_usd",
                F.col("salary_in_usd").cast("double")) \
    .withColumn("salary",
                F.col("salary").cast("double")) \
    .withColumn("work_year",
                F.col("work_year").cast("int"))

print("Jobs cleaned ✓")
jobs_clean.show(5, truncate=True)

Jobs cleaned ✓
+---------+--------------------+--------------------+---------------+--------+-------------+------------------+----------------+---------------+------------+----------------+------------+
|work_year|           job_title|        job_category|salary_currency|  salary|salary_in_usd|employee_residence|experience_level|employment_type|work_setting|company_location|company_size|
+---------+--------------------+--------------------+---------------+--------+-------------+------------------+----------------+---------------+------------+----------------+------------+
|     2023|Data DevOps Engineer|    Data Engineering|            EUR| 88000.0|      95012.0|           Germany|       Mid-level|      Full-time|      Hybrid|         Germany|           L|
|     2023|      Data Architect|Data Architecture...|            USD|186000.0|     186000.0|     United States|          Senior|      Full-time|   In-person|   United States|           M|
|     2023|      Data Architect|Data Architec

In [0]:
# Write Bronze Delta tables
unemp_clean.write.format("delta").mode("overwrite") \
    .saveAsTable("ecommerce.bronze.unemployment_bronze")

jobs_clean.write.format("delta").mode("overwrite") \
    .saveAsTable("ecommerce.bronze.jobs_bronze")

# Optimize
spark.sql("OPTIMIZE ecommerce.bronze.unemployment_bronze")
spark.sql("OPTIMIZE ecommerce.bronze.jobs_bronze")
print("OPTIMIZE done ✓")

# Time travel check
spark.sql("""
    DESCRIBE HISTORY ecommerce.bronze.unemployment_bronze
""").show(3, truncate=False)

print("Bronze layer complete ✓")

OPTIMIZE done ✓
+-------+-------------------+--------------+------------------------+---------------------------------+------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------------+----+------------------+------------------------------------+------------------------+-----------+-----------------+-------------+--------------------------------------------------------------------------------------------------------------------------------------------+------------+--------------------------------------------------+
|version|timestamp          |userId        |userName                |operation                        |operationParameters                                                                                                                                                                             |job |notebook          |queryHistoryStatementId         